In [9]:
PATH = "ideal_data_final.csv"

In [10]:
import pandas as pd

df = pd.read_csv(PATH)

FEATURES = df.columns.tolist()
TARGETS = ["temperature_active1", "temperature_active2", "temperature_active3", "temperature_active4"]

In [11]:
print("loaded shape:", df.shape)

loaded shape: (1440, 18)


In [12]:
import numpy as np

# load
expected = set(FEATURES)
present = set(df.columns)
missing = [c for c in FEATURES if c not in present]
extra = [c for c in df.columns if c not in expected]

# column audit
print("missing features:", missing)
print("Extra columns (kept, just not used unless added):", extra)

# parsing to timestamp
if 'timestamp' in df.columns:
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        print("Timestamp parsed to datetime64")
    except Exception as e:
        print("Timestamp parse failed:", e)

# basic stats & NaNs
print("\nDTypes:\n", df.dtypes.head(20))
nan_report = df[FEATURES if not missing else df.columns].isna().sum().sort_values(ascending=False)
print("\nTop NaN counts:\n", nan_report.head(15))

# sort by time if timestamp exists
if 'timestamp' in df.columns and np.issubdtype(df['timestamp'].dtype, np.datetime64):
    df = df.sort_values('timestamp').reset_index(drop=True)

# head
display(df.head(3))

missing features: []
Extra columns (kept, just not used unless added): []
Timestamp parsed to datetime64

DTypes:
 timestamp              datetime64[ns]
hour                            int64
phase                          object
temperature_active1           float64
temperature_active2           float64
temperature_active3           float64
temperature_active4           float64
temperature_curing1           float64
temperature_curing2           float64
moisture_active1              float64
moisture_active2              float64
moisture_curing1              float64
moisture_curing2              float64
oxygen                        float64
co2                           float64
methane                       float64
methane_ppm                   float64
aeration_on                     int64
dtype: object

Top NaN counts:
 timestamp              0
hour                   0
phase                  0
temperature_active1    0
temperature_active2    0
temperature_active3    0
temperature_active4

,timestamp,hour,phase,temperature_active1,temperature_active2,temperature_active3,temperature_active4,temperature_curing1,temperature_curing2,moisture_active1,moisture_active2,moisture_curing1,moisture_curing2,oxygen,co2,methane,methane_ppm,aeration_on
0,2025-01-01 00:00:00,0,active,34.81,35.74,34.17,35.25,31.69,32.64,58.05,59.93,44.78,45.51,8.80,0.00,0.0,0.0,1
1,2025-01-01 01:00:00,1,active,34.57,35.93,34.27,35.40,31.59,32.64,58.12,60.05,44.75,45.53,8.14,0.01,0.0,0.0,0
2,2025-01-01 02:00:00,2,active,34.89,35.61,34.24,35.11,31.53,32.63,57.92,60.22,44.84,45.59,8.27,0.01,0.0,0.0,0


In [ ]:
# clean, encode, impute

df = df.fillna(method="ffill").fillna(method='bfill')

# one-hot encode phase if present
if 'phase' in df.columns:
    df = pd.get_dummies(df, columns=['phase'], prefix='phase')

# one-hot encode aeration_on as categorical
if 'aeration_on' in df.columns:
    # normalize to clean 1/0 ints
    if np.issubdtype(df['aeration_on'].dtype, np.number):
        aer = df['aeration_on'].fillna(0).astype(float).round().clip(0,1).astype(int)
    elif df['aeration_on'].dtype == bool:
        aer = df['aeration_on']